# L15 · 请求与响应：前后端如何握手

**学习目标**
- 区分「路径参数」和「查询参数」
- 理解常见状态码（200/404/422/500）
- 写出一个带参数路由的 API

**前置依赖**：L13、L14（FastAPI 基础）  
**预计时长**：40 分钟  
**技术栈**：`fastapi`、`uvicorn`、`requests`

---

## 概念讲解：URL 里的两种「变量」

- **路径参数**：写在路径里的必填项，如 `/users/123` 的 `123` 是用户 ID（定位一个资源）
- **查询参数**：写在 `?` 后的可选项，如 `/search?q=猫&page=2`（过滤/分页）

**状态码** 是服务器回的「结果信号」：2xx 成功、4xx 你错了、5xx 我崩了。

## 第一步：路径参数 + 查询参数

In [ ]:
from fastapi import FastAPI
app = FastAPI()

@app.get("/users/{user_id}")
def get_user(user_id: int, verbose: bool = False):
    base = {"id": user_id, "name": f"用户{user_id}"}
    if verbose:
        base["note"] = "verbose 模式返回更多字段"
    return base

## 第二步：启动并分别测试两类参数

In [ ]:
import uvicorn, threading, time, requests
PORT = 8771
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning"), daemon=True).start()
time.sleep(2)
print("路径参数:", requests.get(f"http://127.0.0.1:{PORT}/users/42").json())
print("加查询参数 verbose=true:", requests.get(f"http://127.0.0.1:{PORT}/users/42?verbose=true").json())

## 第三步：404 与状态码

In [ ]:
r = requests.get(f"http://127.0.0.1:{PORT}/users/abc")   # 路径参数不是整数
print("访问 /users/abc 的状态码：", r.status_code, "（422=参数类型不符）")

# 🎯 AHA 顿悟单元格：一个「会查数据库的迷你搜索 API」

运行本单元格（含上方服务）。你得到一个**搜索接口**：用 `?q=关键词&limit=数量` 过滤内存数据，
未找到时返回 404。试试把 `q` 改成「猫」「狗」「不存在的词」，看响应与状态码如何变化。

> 路径/查询参数 + 状态码，就是你在任何 App 里「搜索、翻页、报错」背后的全部机制。

In [ ]:
# ===== 运行我！（需先运行上面的服务定义与启动）=====
from fastapi import FastAPI, HTTPException
import requests, json

db = [{"id": i, "name": n} for i, n in enumerate(["橘猫", "柴犬", "布偶猫", "柯基", "波斯猫"], 1)]
search_app = FastAPI()

@search_app.get("/search")
def search(q: str, limit: int = 3):
    hits = [x for x in db if q in x["name"]]
    if not hits:
        raise HTTPException(status_code=404, detail=f"没找到包含『{q}』的宠物")
    return hits[:limit]

PORT2 = 8772
import uvicorn, threading, time
threading.Thread(target=lambda: uvicorn.run(search_app, host="127.0.0.1", port=PORT2, log_level="warning"), daemon=True).start()
time.sleep(2)

for q in ["猫", "狗", "鱼"]:
    r = requests.get(f"http://127.0.0.1:{PORT2}/search", params={"q": q})
    print(f"  q='{q}' → 状态码 {r.status_code}: {r.json() if r.status_code==200 else r.json()['detail']}")
print("  🔍 搜索 API 跑通：成功返回列表，失败返回 404 + 友好提示。")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：路径参数(必填, 在路径) vs 查询参数(可选, 在?)；`HTTPException` 抛错。  
**易错点**：同一 cell 多次定义 `app` 变量冲突，本方案用独立 `search_app`/PORT2 规避。  
**AHA 机制**：真实搜索 API + 404 友好提示，强「我的服务会智能应答」感。  
**衔接**：L16 数据库（把内存 `db` 换成真实 DB）；L17 鉴权（给搜索加密）。  
**依赖**：`pip install fastapi uvicorn requests`。

# 📚 作业 / 下一步

1. 把 `limit` 改成 1，再搜「猫」，看返回条数变化。
2. 给 `/search` 加一个 `sort` 查询参数，支持按 id 排序。
3. 下一课 **L16 数据库实战：让数据持久化** —— 用 SQLite 让服务重启后数据还在。